# 07_Dashboard
**발표 스킬 자동 분석 시스템 — 대시보드**

| 셀 | 내용 |
|---|---|
| 셀 1 | Drive 마운트 + 패키지 설치 |
| 셀 2 | 모델 로드 (inference_utils) |
| 셀 3 | Gradio 대시보드 실행 |

## 셀 1 — Drive 마운트 + 패키지 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install openai-whisper gradio -q
!apt-get install -y ffmpeg -q

## 셀 2 — 모델 로드

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/presentation_data/')
from inference_utils import *

model, tokenizer, scaler = load_model_and_scaler()

## 셀 3 — Gradio 대시보드

In [ ]:
import gradio as gr

# ── 결과 → HTML 변환 ───────────────────────────────────────────
def result_to_html(result: dict) -> str:
    pred      = result['prediction']
    prob_good = result['prob_good'] * 100
    prob_poor = result['prob_poor'] * 100
    f         = result['features']
    fb        = result['feedback']
    wpm       = result['wpm']

    pred_color = '#2ecc71' if pred == 'Good' else '#e74c3c'
    pred_bg    = '#e8f8f0' if pred == 'Good' else '#fdecea'

    # 카드별 색상
    def bar_color(value, good_fn):
        return '#2ecc71' if good_fn(value) else ('#e67e22' if good_fn(value * 0.7) else '#e74c3c')

    fr_color  = '#2ecc71' if f['filler_ratio'] < 0.03 else ('#e67e22' if f['filler_ratio'] < 0.07 else '#e74c3c')
    vd_color  = '#2ecc71' if f['vocab_diversity'] > 0.70 else ('#e67e22' if f['vocab_diversity'] > 0.50 else '#e74c3c')
    wpm_color = '#2ecc71' if 250 <= wpm <= 350 else ('#e67e22' if (200 <= wpm < 250 or 350 < wpm <= 400) else ('#95a5a6' if wpm == 0 else '#e74c3c'))

    fr_pct  = min(f['filler_ratio'] * 100 / 15 * 100, 100)
    vd_pct  = f['vocab_diversity'] * 100
    wpm_pct = min(wpm / 500 * 100, 100) if wpm > 0 else 0

    # 피드백 카드 이모지
    filler_icon = '✅' if f['filler_ratio'] < 0.03 else ('🟡' if f['filler_ratio'] < 0.07 else '🔴')
    vocab_icon  = '✅' if f['vocab_diversity'] > 0.70 else ('🟡' if f['vocab_diversity'] > 0.50 else '🔴')
    wpm_icon    = 'ℹ️' if wpm == 0 else ('✅' if 250 <= wpm <= 350 else ('🟡' if (200 <= wpm < 250 or 350 < wpm <= 400) else '🔴'))

    html = f"""
    <style>
      @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@400;500;700;900&family=DM+Mono:wght@400;500&display=swap');
      .dash {{ font-family: 'Noto Sans KR', sans-serif; max-width: 780px; margin: 0 auto; padding: 8px; color: #1a1a2e; }}

      /* 헤더 */
      .header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px; }}
      .title {{ font-size: 20px; font-weight: 900; color: #1a1a2e; }}
      .badge {{ display: inline-block; padding: 4px 14px; border-radius: 20px; font-size: 13px; font-weight: 700;
                background: {pred_bg}; color: {pred_color}; border: 1.5px solid {pred_color}; margin-left: 10px; }}
      .prob-box {{ text-align: right; font-size: 13px; line-height: 1.8; }}
      .prob-good {{ color: #2ecc71; font-weight: 700; }}
      .prob-poor {{ color: #e74c3c; font-weight: 700; }}

      /* 메트릭 카드 */
      .cards {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 20px; }}
      .card {{ background: #fff; border-radius: 14px; padding: 16px; box-shadow: 0 2px 12px rgba(0,0,0,0.07); }}
      .card-label {{ font-size: 11px; color: #888; font-weight: 500; margin-bottom: 6px; }}
      .card-value {{ font-family: 'DM Mono', monospace; font-size: 28px; font-weight: 700; margin-bottom: 8px; }}
      .card-bar-bg {{ background: #f0f0f0; border-radius: 4px; height: 5px; margin-bottom: 6px; }}
      .card-bar {{ height: 5px; border-radius: 4px; transition: width 0.6s ease; }}
      .card-status {{ font-size: 11px; color: #aaa; text-align: right; }}
      .card-hint {{ font-size: 11px; color: #bbb; margin-top: 4px; }}

      /* 피드백 섹션 */
      .section-title {{ font-size: 14px; font-weight: 700; color: #555; margin-bottom: 12px; }}
      .fb-card {{ background: #fff; border-radius: 12px; padding: 16px 18px; margin-bottom: 10px;
                  box-shadow: 0 1px 8px rgba(0,0,0,0.06); display: flex; gap: 14px; align-items: flex-start; }}
      .fb-icon {{ font-size: 20px; flex-shrink: 0; margin-top: 2px; }}
      .fb-title {{ font-size: 14px; font-weight: 700; margin-bottom: 4px; }}
      .fb-body {{ font-size: 13px; color: #555; line-height: 1.6; }}
    </style>

    <div class="dash">
      <!-- 헤더 -->
      <div class="header">
        <div>
          <span class="title">발표 품질 분석</span>
          <span class="badge">{pred}</span>
        </div>
        <div class="prob-box">
          <div><span class="prob-good">● Good</span> &nbsp; {prob_good:.0f}%</div>
          <div><span class="prob-poor">● Poor</span> &nbsp; {prob_poor:.0f}%</div>
        </div>
      </div>

      <!-- 메트릭 카드 -->
      <div class="cards">
        <!-- WPM -->
        <div class="card">
          <div class="card-label">🎙 말 속도 (WPM)</div>
          <div class="card-value" style="color:{wpm_color}">{wpm:.0f}</div>
          <div class="card-bar-bg"><div class="card-bar" style="width:{wpm_pct:.0f}%;background:{wpm_color}"></div></div>
          <div class="card-hint">권장: 250~350 WPM</div>
        </div>
        <!-- 필러워드 -->
        <div class="card">
          <div class="card-label">🔤 필러워드 비율</div>
          <div class="card-value" style="color:{fr_color}">{f['filler_ratio']*100:.1f}%</div>
          <div class="card-bar-bg"><div class="card-bar" style="width:{fr_pct:.0f}%;background:{fr_color}"></div></div>
          <div class="card-hint">권장: 3% 미만</div>
        </div>
        <!-- 어휘 다양성 -->
        <div class="card">
          <div class="card-label">📚 어휘 다양성</div>
          <div class="card-value" style="color:{vd_color}">{f['vocab_diversity']:.2f}</div>
          <div class="card-bar-bg"><div class="card-bar" style="width:{vd_pct:.0f}%;background:{vd_color}"></div></div>
          <div class="card-hint">권장: 0.70 이상</div>
        </div>
        <!-- 총 단어 수 -->
        <div class="card">
          <div class="card-label">✏️ 총 단어 수</div>
          <div class="card-value" style="color:#1a1a2e">{f['total_words']}</div>
          <div class="card-bar-bg"><div class="card-bar" style="width:60%;background:#ccc"></div></div>
          <div class="card-hint">평균 단어 길이 {f['avg_word_len']:.1f}자</div>
        </div>
      </div>

      <!-- 항목별 피드백 -->
      <div class="section-title">항목별 피드백</div>

      <div class="fb-card">
        <div class="fb-icon">{filler_icon}</div>
        <div>
          <div class="fb-title">필러워드</div>
          <div class="fb-body">{fb['filler']}</div>
        </div>
      </div>

      <div class="fb-card">
        <div class="fb-icon">{wpm_icon}</div>
        <div>
          <div class="fb-title">말 속도</div>
          <div class="fb-body">{fb['wpm']}</div>
        </div>
      </div>

      <div class="fb-card">
        <div class="fb-icon">{vocab_icon}</div>
        <div>
          <div class="fb-title">어휘 다양성</div>
          <div class="fb-body">{fb['vocab']}</div>
        </div>
      </div>
    </div>
    """
    return html


# ── 분석 함수 ──────────────────────────────────────────────────
def analyze_audio(audio_path):
    if audio_path is None:
        return "<p style='color:#e74c3c;padding:20px'>⚠️ 음성 파일을 업로드해주세요.</p>"
    try:
        text, duration = stt_from_audio(audio_path)
        result = run_inference(text, model, tokenizer, scaler, duration_seconds=duration)
        return result_to_html(result)
    except Exception as e:
        return f"<p style='color:#e74c3c;padding:20px'>❌ 오류: {e}</p>"


def analyze_text(script, duration_min):
    if not script.strip():
        return "<p style='color:#e74c3c;padding:20px'>⚠️ 스크립트를 입력해주세요.</p>"
    try:
        duration_sec = float(duration_min) * 60 if duration_min else 0.0
        result = run_inference(script, model, tokenizer, scaler, duration_seconds=duration_sec)
        return result_to_html(result)
    except Exception as e:
        return f"<p style='color:#e74c3c;padding:20px'>❌ 오류: {e}</p>"


# ── Gradio UI ──────────────────────────────────────────────────
css = """
.gradio-container { background: #f5f5f0 !important; }
footer { display: none !important; }
"""

with gr.Blocks(css=css, title='발표 스킬 분석') as demo:
    gr.Markdown("## 🎤 발표 스킬 자동 분석 시스템")

    with gr.Tabs():
        # 탭 1: 음성 파일
        with gr.Tab('🎙 음성 파일 분석'):
            audio_input = gr.Audio(type='filepath', label='음성 파일 업로드 (.mp3 / .wav)')
            audio_btn   = gr.Button('분석 시작', variant='primary')
            audio_out   = gr.HTML()
            audio_btn.click(fn=analyze_audio, inputs=audio_input, outputs=audio_out)

        # 탭 2: 텍스트 입력
        with gr.Tab('📝 텍스트 직접 입력'):
            text_input = gr.Textbox(
                lines=8,
                placeholder='발표 스크립트를 여기에 붙여넣으세요...',
                label='발표 스크립트'
            )
            dur_input = gr.Number(label='발표 시간 (분) — WPM 계산용, 모르면 비워두세요', value=None)
            text_btn  = gr.Button('분석 시작', variant='primary')
            text_out  = gr.HTML()
            text_btn.click(fn=analyze_text, inputs=[text_input, dur_input], outputs=text_out)

demo.launch(debug=True)